# ColliderFM Diagnostics Explorer

This notebook is the interactive companion to `scripts/plot_diagnostics.py`.

It follows the current calo-only project state and lets you inspect:

1. one raw calorimeter event
2. one point view
3. global, local, and masked views
4. model outputs from a checkpoint
5. pooled embeddings and prototype usage


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from collider_fm.data import ColliderMLDataset
from collider_fm.diagnostics import compute_pca, encode_view, load_checkpoint
from collider_fm.model import create_small_panda_model
from collider_fm.views import (
    CALO_TYPE_NAMES,
    augment_point_view,
    batch_point_views,
    build_point_view_from_event,
    local_crop_point_view,
    mask_point_view,
)

plt.style.use('default')
plt.rcParams['figure.dpi'] = 120


## Configuration

In [ ]:
DETAIL_SPLIT = 'train[0:1]'
REPRESENTATION_SPLIT = 'train[:10]'
DATASET_TYPE = 'ttbar'
PU_CONFIG = 'pu0'
CACHE_DIR = '/mnt/ceph/users/ewulff/data/hf'
MAX_CALO_HITS = 256
CHECKPOINT_PATH = PROJECT_ROOT / 'runs' / 'monday_ssl_baseline' / 'checkpoint.pt'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
print('checkpoint:', CHECKPOINT_PATH)


## Load one event

In [ ]:
detail_dataset = ColliderMLDataset(
    split=DETAIL_SPLIT,
    dataset_type=DATASET_TYPE,
    pu_config=PU_CONFIG,
    object_types=['calo_hits'],
    cache_dir=CACHE_DIR,
)
detail_event = detail_dataset[0]
detail_view = build_point_view_from_event(detail_event, device=torch.device('cpu'), max_calo_hits=MAX_CALO_HITS)
print('num points:', detail_view['coord'].shape[0])
print('feature shape:', detail_view['feat'].shape)


## Plot the raw event

In [ ]:
calo_hits = detail_event['calo_hits']
x = np.asarray(torch.as_tensor(calo_hits['x']).tolist(), dtype=float)
z = np.asarray(torch.as_tensor(calo_hits['z']).tolist(), dtype=float)
energy = np.asarray(torch.as_tensor(calo_hits['energy']).tolist(), dtype=float)
plt.figure(figsize=(7, 5))
plt.scatter(z, x, c=energy, s=3, cmap='inferno', alpha=0.6)
plt.xlabel('z')
plt.ylabel('x')
plt.title('Raw calorimeter event')
plt.colorbar(label='energy')
plt.show()


## Build three view types

In [ ]:
global_view = augment_point_view(detail_view)
local_view = local_crop_point_view(augment_point_view(detail_view))
masked_view = mask_point_view(augment_point_view(detail_view))
print('global hidden:', int(global_view['hidden_mask'].sum().item()))
print('local loss points:', int(local_view['loss_mask'].sum().item()))
print('masked loss points:', int(masked_view['loss_mask'].sum().item()))


In [ ]:
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.bar(['used', 'ignored'], [local_view['loss_mask'].sum().item(), (~local_view['loss_mask']).sum().item()])
plt.title('Local view loss mask')
plt.subplot(1, 2, 2)
plt.bar(['masked', 'visible'], [masked_view['hidden_mask'].sum().item(), (~masked_view['hidden_mask']).sum().item()])
plt.title('Masked view hidden points')
plt.tight_layout()
plt.show()


## Load a checkpoint and inspect outputs

In [ ]:
if DEVICE.type != 'cuda':
    print('CUDA is unavailable, so model-backed cells are skipped.')
else:
    model = create_small_panda_model(device=DEVICE)
    load_checkpoint(model, str(CHECKPOINT_PATH))
    model.eval()
    gpu_view = build_point_view_from_event(detail_event, device=DEVICE, max_calo_hits=MAX_CALO_HITS)
    encoding = encode_view(model, gpu_view)
    for key, value in encoding.items():
        print(key, tuple(value.shape))


## Plot pooled embedding PCA

In [ ]:
if DEVICE.type != 'cuda':
    print('CUDA is unavailable, so PCA is skipped.')
else:
    rep_dataset = ColliderMLDataset(
        split=REPRESENTATION_SPLIT,
        dataset_type=DATASET_TYPE,
        pu_config=PU_CONFIG,
        object_types=['calo_hits'],
        cache_dir=CACHE_DIR,
    )
    rep_views = [
        build_point_view_from_event(rep_dataset[index], device=DEVICE, max_calo_hits=MAX_CALO_HITS)
        for index in range(min(len(rep_dataset), 10))
    ]
    rep_batch = batch_point_views(rep_views)
    rep_encoding = encode_view(model, rep_batch)
    projected = compute_pca(rep_encoding['pooled'].detach().cpu().numpy(), n_components=2)
    plt.figure(figsize=(6, 5))
    plt.scatter(projected[:, 0], projected[:, 1], c=np.arange(projected.shape[0]), cmap='tab10')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.title('Pooled embedding PCA')
    plt.show()
